# core

> Async Google Workspace clients built from Google's Discovery API.

In [ ]:
#| default_exp core

In [ ]:
#| export
from collections.abc import Sized
from fastcore.utils import *
from fastgws.auth import *
from fastspec.spec import SpecParser
from fastspec.oapi import AsyncTransport, OpFunc, _build_groups
from googleapiclient.discovery import build
from googleapiclient.http import BatchHttpRequest

import asyncio, httpx, os, random

In [ ]:
from fastcore.test import *
from fastspec.errors import APIError

import time

In [ ]:
#| export
class GWSObject(AttrDict):
    def __repr__(self):
        keys = 'id name title summary emailAddress threadId documentId spreadsheetId sheetId mimeType kind'.split()
        bits = [f"{k}={self[k]!r}" for k in keys if k in self]
        bits += [f"{k}={len(v)}" for k,v in self.items()
                 if isinstance(v, Sized) and not isinstance(v, (str, bytes)) and k not in keys]
        return f"{type(self).__name__}({', '.join(bits)})"

def gclasses(doc):
    res = {}
    for n,s in doc.get('schemas', {}).items():
        kind = nested_idx(s, 'properties', 'kind', 'default')
        if kind: res[kind] = type(n, (GWSObject,), {})
    return res

def g2obj(x, gcls):
    if isinstance(x, list): return L(x).map(lambda o: g2obj(o, gcls))
    if not isinstance(x, dict): return x
    cls = gcls.get(x.get('kind'), GWSObject)
    return cls({k:g2obj(v, gcls) for k,v in x.items()})

In [ ]:
#| export
@patch
def _retry_after(self:httpx.HTTPStatusError):
    v = self.response.headers.get('retry-after')
    return float(v) if v and v.replace('.','',1).isdigit() else None

class GWSTransport(AsyncTransport):
    def __init__(self, *args, gcls=None, creds=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.gcls,self.creds = gcls or {},creds

    def _request_headers(self, headers=None, *, files=None):
        if self.creds: self.base_headers |= auth_headers(self.creds)
        return super()._request_headers(headers, files=files)

    async def request(self, *args, raw=False, n_retries=5, base=1.0, **kwargs):
        for i in range(n_retries):
            try:
                res = await super().request(*args, raw=raw, **kwargs)
                return res if raw else g2obj(res, self.gcls)
            except httpx.HTTPStatusError as e:
                if i==n_retries-1 or not e.api_error().retryable: raise
                wait = e._retry_after() or base*2**i + random.uniform(0, base)
                await asyncio.sleep(wait)

class GWSOpFunc(OpFunc): pass

class GWSApi:
    service = None

    def __init__(self, service=None, version=None, token=None, creds=None,
                 api_key=None, headers=None, timeout=60.0, doc=None):
        service = ifnone(service, self.service)
        if service is None: raise ValueError('`service` is required')
        self.service,self.version = service,version
        self.doc = doc or self._get_doc(service, version)
        self._gservice = build(service, self.version, credentials=creds, cache_discovery=False)
        self.spec = SpecParser.from_discovery(self.doc)
        self.gcls = gclasses(self.doc)

        hdrs = headers or {}
        if creds or token: hdrs = {**auth_headers(creds, token), **hdrs}
        if api_key is None and not (creds or token): api_key = os.getenv('GOOGLE_API_KEY', os.getenv('GWS_API_KEY'))
        if api_key: hdrs = {'X-Goog-Api-Key': api_key, **hdrs}

        self.transport = GWSTransport(timeout=timeout, base_headers=hdrs, gcls=self.gcls, creds=creds)
        self.ops = [GWSOpFunc(o, self.transport, self.spec.base_url, noop) for o in self.spec.ops]
        self.func_dict = {f"{o.path}:{o.verb.upper()}": o for o in self.ops}
        self.groups = _build_groups(self.ops)
        for k,v in self.groups.items(): setattr(self, k, v)

    def _get_doc(self, service, version=None):
        if version:
            url = f'https://www.googleapis.com/discovery/v1/apis/{service}/{version}/rest'
        else:
            apis = httpx.get('https://discovery.googleapis.com/discovery/v1/apis').json()['items']
            api = first(a for a in apis if a['name'] == service and a.get('preferred'))
            url = api['discoveryRestUrl']
            self.version = api['version']
        return httpx.get(url).json()
    
    def batch_get(self, requests, chunk_sz=50):
        "Batch fetch `requests` (list of (id, HttpRequest) tuples) via googleapiclient batch protocol"
        results = {}
        def _cb(rid, resp, exc):
            if exc: raise exc
            results[rid] = resp
        uri = f'https://www.googleapis.com/batch/{self.service}/{self.version}'
        for chunk in chunked(requests, chunk_sz):
            batch = BatchHttpRequest(callback=_cb, batch_uri=uri)
            for rid, r in chunk: batch.add(r, request_id=rid)
            batch.execute()
        return results

In [ ]:
creds = await oauth_creds(scopes=['https://www.googleapis.com/auth/gmail.readonly',
                                  'https://www.googleapis.com/auth/drive.readonly'],
                          redirect_uri='https://oauth.appapis.org/redirect')

In [ ]:
drive = GWSApi('drive', creds=creds)
fs = await drive.files.list(q="name contains 'fastgws' and trashed=false", page_size=10)
fs, fs.files[0]

({'files': [{'kind': 'drive#file', 'mimeType': 'application/vnd.google-apps.document', 'id': '1ObmgD5GOA9zNZbUwCYeFZH8nUc_MKcJHKFqjPJGnkQs', 'name': 'fastgws test doc'}, {'kind': 'drive#file', 'mimeType': 'application/vnd.google-apps.document', 'id': '1ohlMW4OttiYNExpTR4SfVdDpGwW7qvyJAYY-YcxaYW0', 'name': 'fastgws test doc'}, {'kind': 'drive#file', 'mimeType': 'application/vnd.google-apps.document', 'id': '1bSoqbh12B5pRF4TEA5RqmWiIB6j2RuK9WNM_Q-6Vers', 'name': 'fastgws test doc'}],
  'kind': 'drive#fileList',
  'incompleteSearch': False},
 {'kind': 'drive#file',
  'mimeType': 'application/vnd.google-apps.document',
  'id': '1ObmgD5GOA9zNZbUwCYeFZH8nUc_MKcJHKFqjPJGnkQs',
  'name': 'fastgws test doc'})

In [ ]:
gmail = GWSApi('gmail', creds=creds)
msgs = await gmail.users.messages.list(user_id='me', max_results=10)
len(msgs['messages'])

10

In [ ]:
reqs = [(m.id, gmail._gservice.users().messages().get(userId='me', id=m.id)) for m in msgs.messages]
res = gmail.batch_get(reqs)
len(res)

10

In [ ]:
url = 'https://fast-http-bin.pla.sh/status/'
status = 500

t = time.time()
try: await gmail.transport.request('GET', url + str(status), base=0.05)
except httpx.HTTPStatusError as e:
    assert time.time() - t > 1
    test_eq(e.response.status_code, status)
    test_eq(e.api_error().retryable, True)

In [ ]:
status = 400

t = time.time()
try: await gmail.transport.request('GET', url + str(status), base=0.05)
except httpx.HTTPStatusError as e:
    assert time.time() - t < .5
    test_eq(e.response.status_code, status)
    test_eq(e.api_error().retryable, False)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()